In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import matplotlib.colors as colors
# import holoviews as hv
from matplotlib.patches import Rectangle

from cycler import cycler
import matplotlib as mpl
import os

In [ ]:
from scipy.signal import savgol_filter
import gzip
import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from numpy.fft import rfft, rfftfreq

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.gridspec import GridSpec

from scipy.signal import find_peaks
from scipy.signal import peak_widths
from scipy.optimize import curve_fit

import matplotlib.lines as mlines

import skunk

In [ ]:
plt.rcParams.update({'font.size': 14})

In [ ]:
def load1dFig1(i):
    return np.loadtxt(os.path.join('..', '20221020 HMIA 13 Hall Transparency',  'data', f'{i}', 'data.tsv'))

def load2dFig1(i, num):
    tmp = np.loadtxt(os.path.join('..', '20221020 HMIA 13 Hall Transparency',  'data', f'{i}', 'data.tsv'))
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise   

In [ ]:
# Gain table
q1 = 100.075
q2 = 100.103
q3 = 100.087
q4 = 100.082

s1 = 100.104
s2 = 99.977
s3 = 100.183

# Gain corrections
def gain_corr(runid, lia_no):
    dat = load1dFig1(runid)
    return(np.mean(dat[:, 2*lia_no - 1]), np.std(dat[:, 2*lia_no - 1], ddof=1) / np.sqrt(np.size(dat[:, 2*lia_no - 1])))

qa1_source = gain_corr(107, 1)
sea2_reflected = gain_corr(107, 4)
sea3_reflected = gain_corr(108, 5)
sea2_source = gain_corr(109, 4)
qa1_reflected = gain_corr(109, 1)
sea1_reflected = gain_corr(110, 3)
sea1_source = gain_corr(111, 3)
sea3_source = gain_corr(112, 5)
qa2_reflected = gain_corr(112, 1)
qa2_source = gain_corr(113, 1)
qa3_reflected = gain_corr(113, 2)
qa3_source = gain_corr(115, 2)
qa4_reflected = gain_corr(115, 1)
qa4_source = gain_corr(116, 1)

In [ ]:
transparencies_dict = {}
sem_dict = {}
std_dict = {}
split_t_dict = {}

In [ ]:
# helper scripts
def find_means(v_mat, grid_bounds):
    yindexmins, yindexmaxs, xindexmins, xindexmaxs = grid_bounds
    n_modes = len(yindexmins)
    mean = np.zeros((n_modes, n_modes))
    sem = np.zeros((n_modes, n_modes))
    std = np.zeros((n_modes, n_modes))

    for i in range(n_modes):
        for j in range(n_modes):
            mean[i, j] = np.mean((v_mat)[xindexmins[i]:xindexmaxs[i], yindexmins[j]:yindexmaxs[j]])
            sem[i, j] = np.std(v_mat[xindexmins[i]:xindexmaxs[i], yindexmins[j]:yindexmaxs[j]], ddof=1) / np.sqrt(
                np.size(v_mat[xindexmins[i]:xindexmaxs[i], yindexmins[j]:yindexmaxs[j]]))
            std[i, j] = np.std(v_mat[xindexmins[i]:xindexmaxs[i], yindexmins[j]:yindexmaxs[j]])
            
    return (mean, sem, std) 


def calculate_transparencies(vR_tuple, vT_tuple, vs_tuple, vRtotal_tuple, nu, print_t=True): 
    (meansR, semR, stdR) = vR_tuple
    (meansT, semT, stdT) = vT_tuple
    (meansS, semS, stdS) = vs_tuple
    (vRtotal, vRtotal_sem, vRtotal_std) = vRtotal_tuple
    n_modes = meansR.shape[0]
    
    # calculating transparency based on VR/VR-at-pinch-off, VT/VS, VR/VS, and VT/(VR+VT)
    t_vRvR = np.zeros(n_modes)
    tsem_vRvR = np.zeros(n_modes)
    tstd_vRvR = np.zeros(n_modes)
    t_vTvs = np.zeros(n_modes)
    tsem_vTvs = np.zeros(n_modes)
    tstd_vTvs = np.zeros(n_modes)
    t_vRvs = np.zeros(n_modes)
    tsem_vRvs = np.zeros(n_modes)
    tstd_vRvs = np.zeros(n_modes)
    t_vTvRvT = np.zeros(n_modes)
    tsem_vTvRvT = np.zeros(n_modes)
    tstd_vTvRvT = np.zeros(n_modes)
    for i in range(n_modes):
        t_vRvR[i] = 2 * nu * (1 - meansR[i, i] / vRtotal) - np.sum(t_vRvR[:i])
        tsem_vRvR[i] = np.linalg.norm(np.append(tsem_vRvR,
            2 * nu * np.sqrt((semR[i, i] / meansR[i, i]) ** 2 + (vRtotal_sem / vRtotal) ** 2) * t_vRvR[i]))
        tstd_vRvR[i] = np.linalg.norm(np.append(tstd_vRvR,
            2 * nu * np.sqrt((stdR[i, i] / meansR[i, i]) ** 2 + (vRtotal_std / vRtotal) ** 2) * t_vRvR[i]))

        t_vTvs[i] = 2 * nu * meansT[i, i] / meansS[i, i] - np.sum(t_vTvs[:i])
        tsem_vTvs[i] = np.linalg.norm(np.append(tsem_vTvs,
            2 * nu * np.linalg.norm((semT[i, i] / meansT[i, i], semS[i, i] / meansS[i, i], 
            qa1_reflected[1] / qa1_reflected[0], sea1_reflected[1] / sea1_reflected[0])) * t_vTvs[i]))
        tstd_vTvs[i] = np.linalg.norm(np.append(tstd_vTvs,
            2 * nu * np.linalg.norm((stdT[i, i] / meansT[i, i], semS[i, i] / meansS[i, i], 
            qa1_reflected[1] / qa1_reflected[0], sea1_reflected[1] / sea1_reflected[0])) * t_vTvs[i]))
        
        t_vRvs[i] = 2 * nu * (1 - meansR[i, i] / meansS[i, i]) - np.sum(t_vRvs[:i])
        tsem_vRvs[i] = np.linalg.norm(np.append(tsem_vRvs,
            2 * nu * np.linalg.norm((semR[i, i] / meansR[i, i], semS[i, i] / meansS[i, i], 
            qa2_reflected[1] / qa2_reflected[0], sea1_reflected[1] / sea1_reflected[0])) * t_vRvs[i]))
        tstd_vRvs[i] = np.linalg.norm(np.append(tstd_vRvs,
            2 * nu * np.linalg.norm((stdR[i, i] / meansR[i, i], semS[i, i] / meansS[i, i], 
            qa2_reflected[1] / qa2_reflected[0], sea1_reflected[1] / sea1_reflected[0])) * t_vRvs[i]))
            
        t_vTvRvT[i] = 2 * nu * meansT[i, i] / (meansT[i, i] + meansR[i, i]) - np.sum(t_vTvRvT[:i])
        # skipping sem and std calculation for now
    unicode_list = ["t\u2081", "t\u2082", "t\u2083", "t\u2084"]    
    if print_t:
        print("Calculated using VR / VR(at pinch-off):")    
        for i in range(n_modes):
            print("%s = %.4f \u00B1 %f" % (unicode_list[i], float(t_vRvR[i]), float(tsem_vRvR[i])))
            print(float(meansR[0,0]))
            # vRtotal might be scalar or array-like; cast to float for safe printing
            try:
                print(float(vRtotal))
            except Exception:
                print(vRtotal)
        print("---")
        print("Calculated using VT / Vsource:")    
        for i in range(n_modes):
            print("%s = %.4f \u00B1 %f" % (unicode_list[i], float(t_vTvs[i]), float(tsem_vTvs[i])))
        print("---")
        print("Calculated using VR / Vsource:")    
        for i in range(n_modes):
            print("%s = %.4f \u00B1 %f" % (unicode_list[i], float(t_vRvs[i]), float(tsem_vRvs[i])))
            print(float(meansR[0,0]))
            print(float(meansS[0,0]))
            
        print("---")
        print("Calculated using VT / (VR + VT):")    
        for i in range(n_modes):
            print("%s = %.4f \u00B1 %f" % (unicode_list[i], float(t_vTvRvT[i]), float(tsem_vTvRvT[i])))
            
    return ((t_vRvR, tsem_vRvR, tstd_vRvR), (t_vTvs, tsem_vTvs, tstd_vTvs), (t_vRvs, tsem_vRvs, tstd_vRvs), 
            (t_vTvRvT, tsem_vTvRvT, tstd_vTvRvT))


def v_island_split_t(field_val, dat, grid_bounds, nu):
    vgbot = dat['1'] 
    vgtop = dat['2']
    vs = dat['3']
    vground = dat['5']  * qa1_reflected[0] / qa3_reflected[0]
    vT = dat['7'] * qa1_reflected[0] / sea1_reflected[0] 
    vR = dat['9'] * qa1_reflected[0] / sea2_reflected[0] 
    visland = dat['11'] * qa1_reflected[0] / sea3_reflected[0] 

    yindexmins = grid_bounds[0]
    yindexmaxs = grid_bounds[1]
    xindexmins = grid_bounds[2]
    xindexmaxs = grid_bounds[3]
    ymins = vgbot[0, yindexmins]
    ymaxs = vgbot[0, yindexmaxs]
    xmins = vgtop[xindexmins, 0]
    xmaxs = vgtop[xindexmaxs, 0]

    vR_tuple = find_means(vR, grid_bounds)
    (meansR, semR, stdR) = vR_tuple
    vT_tuple = find_means(vT, grid_bounds)
    (meansT, semT, stdT) = vT_tuple
    visland_tuple = find_means(visland, grid_bounds)
    (meansisland, semisland, stdisland) = visland_tuple
    
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].pcolormesh(vgtop, vgbot, vT/visland, vmin=0, vmax=2)
    ax[0].set_title("Calculated top transparency")
    im = ax[1].pcolormesh(vgtop, vgbot, vR/visland, vmin=0, vmax=2)
    ax[1].set_title("Calculated bottom transparency")
    cbar = fig.colorbar(im, ax=ax[1])
    transparencies = np.zeros((len(ymins), 2))
    for i in range((len(ymins))):
            transparencies[i, :] = np.array([(meansT/meansisland)[i,i] / ((meansR/meansisland)[i,i] + (meansT/meansisland)[i,i] - 1) * nu - np.sum(transparencies[:, 0]),
                                 (meansT/meansisland)[i,i] * nu - np.sum(transparencies[:, 1])])
            ax[0].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[i], ymaxs[i])), "%.3f" % float(transparencies[i, 0]), 
                       horizontalalignment='center', verticalalignment='center')
            ax[1].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[i], ymaxs[i])), "%.3f" % float(transparencies[i, 1]), 
                       horizontalalignment='center', verticalalignment='center')
    split_t_dict[field_val] = (field_val, transparencies)


def transparency_script(field_val, dat, grid_bounds, check_traces, vR_ind, nu, sep=False):
    '''field_val: B-field value
    dat: array of 2D data (use dat = load2d(runid, fast_axis_length))
    grid_bounds: the indices for the bounding boxes of the areas taken for voltages (format is (ymins, ymaxs, xmins, xmaxs))
    check_traces: 0 if ignore, 1 for outer most edge state, etc.
    vR_ind: (x1, y1, x2, y2) vertices of bounding box to take the average of vR over 
    nu: bulk nu value at this field'''

    vgbot = dat['1'] 
    vgtop = dat['2']
    vs = dat['3']
    vground = dat['5']  * qa1_reflected[0] / qa3_reflected[0]
    vT = dat['7'] * qa1_reflected[0] / sea1_reflected[0] 
    vR = dat['9'] * qa1_reflected[0] / sea2_reflected[0] 
    visland = dat['11'] * qa1_reflected[0] / sea3_reflected[0] 

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax = ax.flatten()
    im = ax[0].pcolormesh(vgtop[5:], vgbot[5:], (vT/vs)[5:], vmin=0, vmax=0.5, cmap='inferno')
    cbar = fig.colorbar(im, ax=ax[0])

    im = ax[1].pcolormesh(vgtop[5:], vgbot[5:], (vR/vs)[5:], vmin=0.5, vmax=1, cmap='inferno')
    cbar = fig.colorbar(im, ax=ax[1])

    for axi in ax[:2]:
        axi.set_xlabel("Top QPC (V)")
    ax[0].set_ylabel("Bottom QPC (V)")
    ax[0].set_title("Transmitted")
    ax[1].set_title("Reflected")
    plt.tight_layout() 

    yindexmins = grid_bounds[0]
    yindexmaxs = grid_bounds[1]
    xindexmins = grid_bounds[2]
    xindexmaxs = grid_bounds[3]
    ymins = vgbot[0, yindexmins]
    ymaxs = vgbot[0, yindexmaxs]
    xmins = vgtop[xindexmins, 0]
    xmaxs = vgtop[xindexmaxs, 0]
    dx = np.abs(xmaxs - xmins)
    dy = np.abs(ymaxs - ymins)
    colors = ['tab:red', 'purple', 'tab:green', 'tab:blue']

    if check_traces !=0 :
        fig2, ax2 = plt.subplots(1, 2, figsize=(10, 4))
        for i in range(yindexmins[check_traces-1], yindexmaxs[check_traces-1]):
            ax2[0].plot(vgtop[5:, i], (vR)[5:, i], label='Vgbot='+str(vgbot[0,i]))
        ax[1].axhline(vgbot[0, yindexmins[check_traces-1]], ls='--', lw=1, color='black')
        ax[1].axhline(vgbot[0, yindexmaxs[check_traces-1]], ls='--', lw=1, color='black')
        for i in range(xindexmins[check_traces-1], xindexmaxs[check_traces-1]):
            ax2[1].plot(vgbot[i], (vR)[i], label='Vgtop='+str(vgtop[i]))
        ax[1].axvline(vgtop[xindexmins[check_traces-1], 0], ls='--', lw=1, color='black')
        ax[1].axvline(vgtop[xindexmaxs[check_traces-1], 0], ls='--', lw=1, color='black')
        ax[1].add_patch(Rectangle((vgtop[vR_ind[0], 0], vgbot[0, vR_ind[1]]), np.abs(vgtop[vR_ind[2], 0]-vgtop[vR_ind[0], 0]),
                                  np.abs(vgbot[0, vR_ind[3]]-vgbot[0, vR_ind[1]]),
                                  edgecolor='pink', fill=None, ls='--', lw=1))
        plt.tight_layout() 
    for i in range(len(ymins)):
        ax[1].add_patch(Rectangle((xmins[i], ymaxs[i]), dx[i], dy[i], edgecolor=colors[i], fill=None, ls='--', lw=1))


    (vRtotal, vRtotal_sem, vRtotal_std) = find_means(vR, ([vR_ind[3]], [vR_ind[1]], [vR_ind[0]], [vR_ind[2]]))
    vRtotal_tuple = (vRtotal[0, 0], vRtotal_sem[0, 0], vRtotal_std[0, 0])
#     print(vRtotal_tuple)
    vR_tuple = find_means(vR, grid_bounds)
    (meansR, semR, stdR) = vR_tuple
    vT_tuple = find_means(vT, grid_bounds)
    (meansT, semT, stdT) = vT_tuple
    vs_tuple = find_means(vs, grid_bounds)
    (meansS, semS, stdS) = vs_tuple

    for i in range((len(ymins))):
        for j in range((len(ymins))):
            if i ==j:
                ax[1].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % float(meansR[i,j] / vRtotal), 
                       horizontalalignment='center', verticalalignment='center', color=colors[i])
                ax[0].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % float(meansT[i,j] / meansS[i,j]), 
                       horizontalalignment='center', verticalalignment='center', color='white')
            else:
                ax[1].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % float(meansR[i,j] / vRtotal), 
                       horizontalalignment='center', verticalalignment='center')
                ax[0].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % float(meansT[i,j] / meansS[i,j]), 
                       horizontalalignment='center', verticalalignment='center', color='white')


    ((t_vRvR, tsem_vRvR, tstd_vRvR), (t_vTvs, tsem_vTvs, tstd_vTvs), (t_vRvs, tsem_vRvs, tstd_vRvs), 
            (t_vTvRvT, tsem_vTvRvT, tstd_vTvRvT)) = calculate_transparencies(vR_tuple, vT_tuple, vs_tuple, vRtotal_tuple, nu, print_t=True)
    
    if check_traces == 0:
        transparencies_dict['B='+str(field_val)+' T, nu<=2'] = (field_val, np.array([t_vRvR, t_vTvs, t_vRvs, t_vTvRvT]))
        sem_dict['B='+str(field_val)+' T, nu<=2'] = (field_val, np.array([tsem_vRvR, tsem_vTvs, tsem_vRvs, tsem_vTvRvT]))
        std_dict['B='+str(field_val)+' T, nu<=2'] = (field_val, np.array([tstd_vRvR, tstd_vTvs, tstd_vRvs, tstd_vTvRvT]))
        
    if sep: 
        v_island_split_t(field_val, dat, grid_bounds, nu)


In [ ]:
def transparency_script(field_val, dat, grid_bounds, check_traces, vR_ind, nu, sep=False):
    '''field_val: B-field value
    dat: array of 2D data (use dat = load2d(runid, fast_axis_length))
    grid_bounds: the indices for the bounding boxes of the areas taken for voltages (format is (ymins, ymaxs, xmins, xmaxs))
    check_traces: 0 if ignore, 1 for outer most edge state, etc.
    vR_ind: (x1, y1, x2, y2) vertices of bounding box to take the average of vR over 
    nu: bulk nu value at this field'''

    vgbot = dat['1'] 
    vgtop = dat['2']
    vs = dat['3']
    vground = dat['5']  * qa1_reflected[0] / qa3_reflected[0]
    vT = dat['7'] * qa1_reflected[0] / sea1_reflected[0] 
    vR = dat['9'] * qa1_reflected[0] / sea2_reflected[0] 
    visland = dat['11'] * qa1_reflected[0] / sea3_reflected[0] 

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax = ax.flatten()
    im = ax[0].pcolormesh(vgtop[5:], vgbot[5:], (vT/vs)[5:], vmin=0, vmax=0.5, cmap='inferno')
    cbar = fig.colorbar(im, ax=ax[0])

    im = ax[1].pcolormesh(vgtop[5:], vgbot[5:], (vR/vs)[5:], vmin=0.5, vmax=1, cmap='inferno')
    cbar = fig.colorbar(im, ax=ax[1])

    for axi in ax[:2]:
        axi.set_xlabel("Top QPC (V)")
    ax[0].set_ylabel("Bottom QPC (V)")
    ax[0].set_title("Transmitted")
    ax[1].set_title("Reflected")
    plt.tight_layout() 

    yindexmins = grid_bounds[0]
    yindexmaxs = grid_bounds[1]
    xindexmins = grid_bounds[2]
    xindexmaxs = grid_bounds[3]
    ymins = vgbot[0, yindexmins]
    ymaxs = vgbot[0, yindexmaxs]
    xmins = vgtop[xindexmins, 0]
    xmaxs = vgtop[xindexmaxs, 0]
    dx = np.abs(xmaxs - xmins)
    dy = np.abs(ymaxs - ymins)
    colors = ['tab:red', 'purple', 'tab:green', 'tab:blue']

    if check_traces !=0 :
        fig2, ax2 = plt.subplots(1, 2, figsize=(10, 4))
        for i in range(yindexmins[check_traces-1], yindexmaxs[check_traces-1]):
            ax2[0].plot(vgtop[5:, i], (vR)[5:, i], label='Vgbot='+str(vgbot[0,i]))
        ax[1].axhline(vgbot[0, yindexmins[check_traces-1]], ls='--', lw=1, color='black')
        ax[1].axhline(vgbot[0, yindexmaxs[check_traces-1]], ls='--', lw=1, color='black')
        for i in range(xindexmins[check_traces-1], xindexmaxs[check_traces-1]):
            ax2[1].plot(vgbot[i], (vR)[i], label='Vgtop='+str(vgtop[i]))
        ax[1].axvline(vgtop[xindexmins[check_traces-1], 0], ls='--', lw=1, color='black')
        ax[1].axvline(vgtop[xindexmaxs[check_traces-1], 0], ls='--', lw=1, color='black')
        ax[1].add_patch(Rectangle((vgtop[vR_ind[0], 0], vgbot[0, vR_ind[1]]), np.abs(vgtop[vR_ind[2], 0]-vgtop[vR_ind[0], 0]),
                                  np.abs(vgbot[0, vR_ind[3]]-vgbot[0, vR_ind[1]]),
                                  edgecolor='pink', fill=None, ls='--', lw=1))
        plt.tight_layout() 
    for i in range(len(ymins)):
        ax[1].add_patch(Rectangle((xmins[i], ymaxs[i]), dx[i], dy[i], edgecolor=colors[i], fill=None, ls='--', lw=1))


    (vRtotal, vRtotal_sem, vRtotal_std) = find_means(vR, ([vR_ind[3]], [vR_ind[1]], [vR_ind[0]], [vR_ind[2]]))
    vRtotal_tuple = (vRtotal[0, 0], vRtotal_sem[0, 0], vRtotal_std[0, 0])
#     print(vRtotal_tuple)
    vR_tuple = find_means(vR, grid_bounds)
    (meansR, semR, stdR) = vR_tuple
    vT_tuple = find_means(vT, grid_bounds)
    (meansT, semT, stdT) = vT_tuple
    vs_tuple = find_means(vs, grid_bounds)
    (meansS, semS, stdS) = vs_tuple

    for i in range((len(ymins))):
        for j in range((len(ymins))):
            if i ==j:
                print(vRtotal.item())
                ax[1].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % ((meansR[i,j].item() / vRtotal.item())), 
                       horizontalalignment='center', verticalalignment='center', color=colors[i])
                ax[0].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % ((meansT[i,j].item() / meansS[i,j].item())), 
                       horizontalalignment='center', verticalalignment='center', color='white')
            else:
                ax[1].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % ((meansR[i,j].item() / vRtotal.item())), 
                       horizontalalignment='center', verticalalignment='center')
                ax[0].text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % ((meansT[i,j].item() / meansS[i,j].item())), 
                       horizontalalignment='center', verticalalignment='center', color='white')


    ((t_vRvR, tsem_vRvR, tstd_vRvR), (t_vTvs, tsem_vTvs, tstd_vTvs), (t_vRvs, tsem_vRvs, tstd_vRvs), 
            (t_vTvRvT, tsem_vTvRvT, tstd_vTvRvT)) = calculate_transparencies(vR_tuple, vT_tuple, vs_tuple, vRtotal_tuple, nu, print_t=True)
    
    if check_traces == 0:
        transparencies_dict['B='+str(field_val)+' T, nu<=2'] = (field_val, np.array([t_vRvR, t_vTvs, t_vRvs, t_vTvRvT]))
        sem_dict['B='+str(field_val)+' T, nu<=2'] = (field_val, np.array([tsem_vRvR, tsem_vTvs, tsem_vRvs, tsem_vTvRvT]))
        std_dict['B='+str(field_val)+' T, nu<=2'] = (field_val, np.array([tstd_vRvR, tstd_vTvs, tstd_vRvs, tstd_vTvRvT]))
        
    if sep: 
        v_island_split_t(field_val, dat, grid_bounds, nu)

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(10, 4))

transparency_script(3.1, load2dFig1(46, 51), ([29, 12], [37, 25], [26, 40], [33, 50]), 0, (5, 50, 21, 3), 6)
transparency_script(3, load2dFig1(47, 51), ([30, 14], [37, 27], [25, 40], [33, 50]), 0, (5, 50, 20, 2), 6, sep=True)
transparency_script(2.9, load2dFig1(49, 51), ([30, 14], [37, 27], [25, 40], [33, 50]), 0, (5, 50, 20, 2), 6, sep=True)

transparency_script(3.5, load2dFig1(55, 131), ([117, 96], [127, 114], [32, 50], [40, 70]), 0, (8, 135, 25, 2), 5, sep=True)
# B=3.6 T
transparency_script(3.6, load2dFig1(56, 81), ([117, 96], [127, 112], [33, 50], [40, 67]), 0, (12, 130, 25, 10), 5, sep=True)
# B=3.7 T
transparency_script(3.7, load2dFig1(57, 81), ([117, 96], [125, 112], [33, 50], [40, 67]), 0, (12, 130, 25, 10), 5, sep=True)
# B=3.8 T
transparency_script(3.8, load2dFig1(58, 81), ([117, 93], [126, 110], [34, 52], [41, 70]), 0, (12, 130, 25, 10), 5, sep=True)

dat = load2dFig1(61, 61)
dat2 = load2dFig1(62, 61)
dat3 = {}
for d in dat.keys():
    dat3[d] = np.append(dat[d], dat2[d], axis=1)
    
transparency_script(4.3, dat3, ([57, 40], [62, 51], [13, 25], [20, 31]), 0, (7, 66, 9, 3), 4, sep=True)

# B=4.4 T
transparency_script(4.4, load2dFig1(64, 96), ([60, 30], [73, 50], [29, 50], [36, 61]), 0, (7, 79, 16, 4), 4, sep=True)
# B=4.5 T
transparency_script(4.5, load2dFig1(66, 96), ([60, 30], [73, 50], [27, 50], [37, 61]), 0, (7, 79, 16, 4), 4, sep=True)
# B=4.6 T
transparency_script(4.6, load2dFig1(68, 96), ([60, 30], [73, 50], [29, 51], [36, 61]), 0, (7, 79, 16, 4), 4, sep=True)
# B = 4.7 T
transparency_script(4.7, load2dFig1(70, 96), ([60, 25], [69, 47], [29, 53], [41, 68]), 0, (7, 79, 17, 4), 4, sep=True)
# B=4.8 T
transparency_script(4.8, load2dFig1(72, 96), ([60, 25], [71, 47], [29, 56], [44, 72]), 0, (7, 79, 17, 4), 4, sep=True)
# B = 4.9 T
transparency_script(4.9, load2dFig1(74, 96), ([58, 20], [71, 45], [29, 58], [40, 74]), 0, (7, 79, 18, 4), 4, sep=True)
# B = 5 T 
# transparency_script(5, load2dFig1(76, 96), ([58, 15], [70, 44], [29, 59], [43, 76]), 0, (7, 79, 18, 4), 4, sep=True)
transparency_script(5.5, load2dFig1(83, 93), ([81, 48, 9], [90, 70, 26], [19, 50, 80], [33, 65, 92]), 0, (7, 110, 11, 4), 3, sep=True)
transparency_script(5.65, load2dFig1(85, 93), ([81, 46, 9], [93, 68, 24], [19, 50, 80], [33, 65, 92]), 0, (7, 110, 11, 4), 3, sep=True)
transparency_script(5.8, load2dFig1(87, 93), ([81, 43, 7], [93, 66, 23], [21, 53, 80], [38, 65, 90]), 0, (7, 110, 11, 4), 3, sep=True)
transparency_script(5.95, load2dFig1(89, 93), ([81, 35, 4], [93, 65, 21], [21, 50, 80], [39, 68, 90]), 0, (7, 110, 11, 4), 3, sep=True)
transparency_script(6.1, load2dFig1(91, 93), ([77, 36, 4], [93, 64, 18], [23, 50, 80], [41, 68, 90]), 0, (7, 110, 12, 4), 3, sep=True)
transparency_script(6.25, load2dFig1(93, 93), ([77, 32, 4], [93, 60, 18], [23, 50, 83], [41, 68, 90]), 0, (7, 110, 12, 4), 3, sep=True)
transparency_script(6.4, load2dFig1(95, 93), ([75, 28, 3], [93, 58, 16], [24, 52, 84], [41, 72, 90]), 0, (7, 110, 12, 4), 3, sep=True)
transparency_script(6.55, load2dFig1(97, 93), ([75, 28, 3], [93, 56, 16], [25, 52, 85], [41, 72, 90]), 0, (7, 110, 12, 4), 3, sep=True)
transparency_script(6.7, load2dFig1(99, 93), ([75, 28, 2], [93, 56, 14], [25, 52, 86], [41, 72, 91]), 0, (7, 110, 12, 4), 3, sep=True)
dat = load2dFig1(61, 61)
dat2 = load2dFig1(62, 61)
dat3 = {}
for d in dat.keys():
    dat3[d] = np.append(dat[d], dat2[d], axis=1)
    
transparency_script(4.3, dat3, ([57, 40], [62, 51], [13, 25], [20, 31]), 0, (7, 66, 9, 3), 4, sep=True)

# B=4.4 T
transparency_script(4.4, load2dFig1(64, 96), ([60, 30], [73, 50], [29, 50], [36, 61]), 0, (7, 79, 16, 4), 4, sep=True)
# B=4.5 T
transparency_script(4.5, load2dFig1(66, 96), ([60, 30], [73, 50], [27, 50], [37, 61]), 0, (7, 79, 16, 4), 4, sep=True)
# B=4.6 T
transparency_script(4.6, load2dFig1(68, 96), ([60, 30], [73, 50], [29, 51], [36, 61]), 0, (7, 79, 16, 4), 4, sep=True)
# B = 4.7 T
transparency_script(4.7, load2dFig1(70, 96), ([60, 25], [69, 47], [29, 53], [41, 68]), 0, (7, 79, 17, 4), 4, sep=True)
# B=4.8 T
transparency_script(4.8, load2dFig1(72, 96), ([60, 25], [71, 47], [29, 56], [44, 72]), 0, (7, 79, 17, 4), 4, sep=True)
# B = 4.9 T
transparency_script(4.9, load2dFig1(74, 96), ([58, 20], [71, 45], [29, 58], [40, 74]), 0, (7, 79, 18, 4), 4, sep=True)
# B = 5 T 
# transparency_script(5, load2dFig1(76, 96), ([58, 15], [70, 44], [29, 59], [43, 76]), 0, (7, 79, 18, 4), 4, sep=True)
# B = 5.5 T (v=3)
transparency_script(5.5, load2dFig1(83, 93), ([81, 48, 9], [90, 70, 26], [19, 50, 80], [33, 65, 92]), 0, (7, 110, 11, 4), 3, sep=True)

transparency_script(5.65, load2dFig1(85, 93), ([81, 46, 9], [93, 68, 24], [19, 50, 80], [33, 65, 92]), 0, (7, 110, 11, 4), 3, sep=True)
# B = 5.8 T
transparency_script(5.8, load2dFig1(87, 93), ([81, 43, 7], [93, 66, 23], [21, 53, 80], [38, 65, 90]), 0, (7, 110, 11, 4), 3, sep=True)
# B = 5.95 T
transparency_script(5.95, load2dFig1(89, 93), ([81, 35, 4], [93, 65, 21], [21, 50, 80], [39, 68, 90]), 0, (7, 110, 11, 4), 3, sep=True)
# B = 6.1 T
transparency_script(6.1, load2dFig1(91, 93), ([77, 36, 4], [93, 64, 18], [23, 50, 80], [41, 68, 90]), 0, (7, 110, 12, 4), 3, sep=True)
# B = 6.25 T
transparency_script(6.25, load2dFig1(93, 93), ([77, 32, 4], [93, 60, 18], [23, 50, 83], [41, 68, 90]), 0, (7, 110, 12, 4), 3, sep=True)

# B = 6.4 T
transparency_script(6.4, load2dFig1(95, 93), ([75, 28, 3], [93, 58, 16], [24, 52, 84], [41, 72, 90]), 0, (7, 110, 12, 4), 3, sep=True)

# B = 6.55 T
transparency_script(6.55, load2dFig1(97, 93), ([75, 28, 3], [93, 56, 16], [25, 52, 85], [41, 72, 90]), 0, (7, 110, 12, 4), 3, sep=True)
# B = 6.7 T
transparency_script(6.7, load2dFig1(99, 93), ([75, 28, 2], [93, 56, 14], [25, 52, 86], [41, 72, 91]), 0, (7, 110, 12, 4), 3, sep=True)
# B = 8 T
transparency_script(8, load2dFig1(121, 69), ([45, 10], [72, 25], [30, 52], [45, 65]), 0, (6, 90, 12, 5), 2, sep=True)
# B = 8.15 T
transparency_script(8.15, load2dFig1(123, 69), ([40, 8], [72, 22], [30, 58], [45, 65]), 0, (6, 90, 12, 5), 2, sep=True)
# B=8.3 T
transparency_script(8.3, load2dFig1(125, 69), ([40, 6], [72, 22], [30, 58], [45, 66]), 0, (6, 90, 12, 5), 2, sep=True)
# B = 8.45 T
transparency_script(8.45, load2dFig1(127, 69), ([40, 6], [72, 22], [31, 58], [45, 66]), 0, (6, 90, 12, 5), 2, sep=True)

# B = 8.6 T
transparency_script(8.6, load2dFig1(129, 69), ([37, 6], [66, 21], [31, 57], [45, 66]), 0, (6, 90, 12, 5), 2, sep=True)

# B = 8.75 T
transparency_script(8.75, load2dFig1(131, 69), ([37, 6], [66, 21], [33, 59], [44, 66]), 0, (6, 90, 12, 5), 2, sep=True)
# B = 8.9 T
transparency_script(8.9, load2dFig1(133, 69), ([37, 6], [66, 21], [34, 59], [44, 66]), 0, (6, 90, 12, 5), 2, sep=True)
# B = 7.9T
transparency_script(7.9, load2dFig1(137, 69), ([39, 6], [67, 21], [32, 56], [46, 66]), 0, (6, 90, 12, 5), 2, sep=True)
# B=9 T
transparency_script(9, load2dFig1(157, 69), ([39, 6], [67, 21], [35, 58], [48, 67]), 0, (6, 90, 12, 5), 2, sep=True)
# with AC excitation = 2.5 nA
# transparency_script(9, load2dFig1(158, 69), ([39, 6], [67, 21], [35, 58], [48, 67]), 0, (6, 90, 12, 5), 2)
# B = 9.1 T
transparency_script(9.1, load2dFig1(161, 69), ([39, 6], [67, 21], [35, 58], [48, 67]), 0, (6, 90, 12, 5), 2, sep=True)
# B = 9.2 T 
transparency_script(9.2, load2dFig1(163, 69), ([39, 6], [60, 20], [36, 59], [48, 67]), 0, (6, 90, 12, 5), 2, sep=True)
# B = 9.3 T
transparency_script(9.3, load2dFig1(165, 69), ([39, 6], [60, 20], [36, 59], [48, 67]), 0, (6, 90, 12, 5), 2, sep=True)
# B = 9.4 T
transparency_script(9.4, load2dFig1(167, 69), ([39, 6], [65, 20], [35, 59], [48, 67]), 0, (6, 90, 12, 5), 2, sep=True)
# B = 9.5 T
transparency_script(9.5, load2dFig1(171, 69), ([39, 6], [65, 20], [35, 59], [48, 67]), 0, (6, 70, 12, 5), 2, sep=True)
# B = 9.6 T
transparency_script(9.6, load2dFig1(173, 69), ([39, 6], [65, 19], [35, 59], [48, 67]), 0, (6, 90, 12, 5), 2, sep=True)

In [ ]:
# %%
# fig, ax = plt.subplots(2, 3, figsize=(10, 8),gridspec_kw={"width_ratios":[1,1, 0.001],"height_ratios":[1, 1]})
# ax1 = ax[0,0]
# ax2 = ax[0,1]
# ax3 = ax[1,0]
# ax4 = ax[1,1]
from matplotlib.gridspec import GridSpec

fig1 = plt.figure(figsize=(12, 12),constrained_layout=True)
gs = fig1.add_gridspec(4, 6,width_ratios=(2,2,1,1,1, 0.2))
# gs = GridSpec(3, 3, figure=fig1)
f1_ax1 = fig1.add_subplot(gs[:3, 0:2])
# f3_ax1.set_title('gs[0, :3]')
f1_ax3 = fig1.add_subplot(gs[3:4, 0:2])
f1_ax2cbar = fig1.add_subplot(gs[0:2, 5])
# plt.setp(f4_ax2.get_yticklabels(), visible=False)
# f3_ax2.set_title('gs[0, 3:]')
f1_ax2 = fig1.add_subplot(gs[:2,2:5])
# f3_ax3.set_title('gs[1,:2]')
f1_ax4 = fig1.add_subplot(gs[2:4,2:6])

f1_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=f1_ax1.transAxes)

f1_ax2.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=f1_ax2.transAxes)

f1_ax3.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=f1_ax3.transAxes)

f1_ax4.text(-0.1, 1, "(d)", fontsize=14, va="bottom", ha="right", transform=f1_ax4.transAxes)


colors = ['b', 'r', 'k', 'tab:purple']
markers = ['h', '>', 'P']

f1_ax41 = inset_axes(f1_ax4,
                    width="30%", # width = 30% of parent_bbox
                    height="30%", # height : 1 inch
                    bbox_to_anchor=(0.72,0.6,0.72, 0.72), bbox_transform=f1_ax4.transAxes,
                    loc=3)

f1_ax41.set_ylim(0.99,1)
f1_ax41.set_yticks(np.linspace(0.99,1,5))
f1_ax41.set_yticklabels(['0.99', '0.9925', '0.995', '0.9975', '1'])
f1_ax41.set_xlabel('B (T)')
f1_ax41.grid(lw=0.4, ls='--')
# fig, ax = plt.subplots(figsize=(9, 5))#, dpi=500)
for d in transparencies_dict.keys():
    B, ts = transparencies_dict[d]
    _, sems = sem_dict[d]
    for i in range(ts.shape[1]):
        f1_ax4.errorbar(B, ts[0, i], yerr=sems[0, i], marker=markers[i], color=colors[i], ms=5, lw=1) # VR/VR
        if B>7.5 and i==0:
            f1_ax41.errorbar(B, ts[0, i], yerr=sems[0, i], marker=markers[i], color=colors[i], ms=5, lw=1) # VR/VR
        # f1_ax4.errorbar(B, ts[1, i], yerr=sems[1, i], marker='x', color=colors[i], ms=4, lw=0.5) # VT/Vs
        # f1_ax4.errorbar(B, ts[2, i], yerr=sems[2, i], marker='s', color=colors[i], ms=4, lw=0.5) # VR/Vs
        # f1_ax4.errorbar(B, ts[3, i], yerr=sems[3, i], marker='^', color=colors[i], ms=4, lw=0.5) # VT/(VR+VT)
f1_ax4.set_xlabel("B (T)")
f1_ax4.set_ylabel("Transparency")
f1_ax4.grid(lw=0.4)
# f1_ax4.set_ylim(0.98,1)
ax2 = f1_ax4.twinx()



magfieldsweeps = [118, 130, 132, 150, 159, 160, 162, 164, 166, 170, 172, 175]
bmean = np.zeros((len(magfieldsweeps)))
for i,d in enumerate(magfieldsweeps):
    dat = load1dFig1(d)
    b = dat[:, 1]
    bmean[i] = np.mean(b)
    vsource = dat[:, 2]
    vground = dat[:, 4]
    vT = dat[:, 6]
    vR = dat[:, 8]
    visland = dat[:, 10]
    ax2.plot(b, 1.01*vsource / (0.5 / 100e6) / 25813, color='tab:purple', lw=1)
#     # ax2.plot(b, scale_factor * (vT + vR) * 1e6, color='tab:purple', ls='--')
    # ax2.plot(b, 1.01 * vground / (0.5 / 100e6) / 100, color='tab:red', lw=1)

# f1_ax4.axhline(0.998,color='k',ls='--')       
ax2.set_ylabel(r"V$_\mathrm{S}/I_0$ ($h/e^2$)", color='tab:purple')
ax2.tick_params(axis='y', colors='tab:purple')
ax2.set_ylim(0, 1)
ax2.axhline(1/2, color='tab:purple', lw=0.5, ls='--')
ax2.axhline(1/3, color='tab:purple', lw=0.5, ls='--')
ax2.axhline(1/4, color='tab:purple', lw=0.5, ls='--')
ax2.axhline(1/5, color='tab:purple', lw=0.5, ls='--')
ax2.axhline(1/6, color='tab:purple', lw=0.5, ls='--')
ax2.set_yticks([1/2, 1/3, 1/4, 1/5, 1/6])
ax2.set_yticklabels([f'1/{s}' for s in range(2,7)])



# legend
leg_circle = mlines.Line2D([], [], color='black', marker='o', ls='None', markersize=5, label='using VR/VR(pinch-off)')
leg_x = mlines.Line2D([], [], color='black', marker='x', ls='None', markersize=5, label='using VT/Vsource')
leg_square = mlines.Line2D([], [], color='black', marker='s', ls='None', markersize=5, label='using VR/Vsource')
leg_tri = mlines.Line2D([], [], color='black', marker='^', ls='None', markersize=5, label='using VT/(VR+VT)')
leg_blue = mlines.Line2D([], [], color='b', marker='h', ls='None', label='i=1')
leg_orange = mlines.Line2D([], [], color='r', marker='>', ls='None', label='i=2')
leg_green = mlines.Line2D([], [], color='k', marker='P', ls='None', label='i=3')
leg_purple = mlines.Line2D([], [], color='tab:purple', marker='None', ls='-', label='$V_S$')
leg_purpledash = mlines.Line2D([], [], color='tab:purple', marker='None', ls='--', label='VR + VT')
leg_red = mlines.Line2D([], [], color='tab:red', marker='None', ls='-', label='Vground (A.U.)')
# ax.legend(handles=[leg_circle, leg_x, leg_square, leg_tri, leg_blue, leg_orange, leg_green, leg_purple, leg_red],
        #   loc=(1.14, 0.2), fontsize=8)
f1_ax4.legend(handles=[leg_blue, leg_orange, leg_green, leg_purple],
          loc=(0.75, 0.10), fontsize=11)
# f1_ax4.legend(handles=[leg_blue, leg_purple],
        #   loc=(0.75, 0.04), fontsize=11)

# gray rectangles for QH
y1, y2 = f1_ax4.get_ylim()
x1, x2 = f1_ax4.get_xlim()
nus = [(2.9, 3.15), (3.5, 3.78), (4.25, 4.95), (5.48, 6.8), (7.9, x2)]
for nu in nus:
    lb, ub = nu
    f1_ax4.add_patch(Rectangle((lb, y1), ub-lb, y2-y1, edgecolor='None', facecolor=(0.3, 0.3, 0.3, 0.1)))
ax2.axes.arrow(8, 0.45, 0.4, 0, lw=0.5, width=0.01, color='tab:purple')
ax0 = f1_ax4.secondary_xaxis('top')
ax0.set_xticks(np.mean(nus[:][:], axis=1))
ax0.set_xticklabels(['6', '5', '4', '3', '2'])
ax0.set_xlabel('Filling factor \u03bd')
# plt.tight_layout()
# fig.savefig("TransparencyvsBvsedgemode.eps", dpi=300)

# Overwrite using file path to my svg
# Can also use a string that contains the SVG

# transparency_script(5.5, load2d(83, 93), ([81, 48, 9], [90, 70, 26], [19, 50, 80], [33, 65, 92]), 0, (7, 110, 11, 4), 3, fig1, f1_ax2, sep=False)



field_val = 5.5
grid_bounds = ([81, 48, 9], [90, 70, 26], [19, 50, 80], [33, 65, 92])
check_traces = 0
vR_ind = (7, 110, 11, 4)
nu = 3

fig = fig1
ax = f1_ax2

dat = load2dFig1(83, 93)
vgbot = dat['1'] 
vgtop = dat['2']
vs = dat['3']
vground = dat['5']  * qa1_reflected[0] / qa3_reflected[0]
vT = dat['7'] * qa1_reflected[0] / sea1_reflected[0] 
vR = dat['9'] * qa1_reflected[0] / sea2_reflected[0] 
visland = dat['11'] * qa1_reflected[0] / sea3_reflected[0] 

# fig, ax = plt.subplots(1, 2, figsize=(10, 4))
# ax = ax.flatten()
# ax0.pcolormesh(vgtop[5:], vgbot[5:], (vT/vs)[5:], vmin=0, vmax=1.1, cmap='inferno')
im = ax.pcolormesh(vgtop[5:], vgbot[5:], (vR/vs)[5:], vmin=0.5, vmax=1, cmap='inferno', rasterized=True)
cbar = fig.colorbar(im, cax=f1_ax2cbar)
cbar.ax.set_ylabel('$V_R/V_{R,0}$')


yindexmins = grid_bounds[0]
yindexmaxs = grid_bounds[1]
xindexmins = grid_bounds[2]
xindexmaxs = grid_bounds[3]
ymins = vgbot[0, yindexmins]
ymaxs = vgbot[0, yindexmaxs]
xmins = vgtop[xindexmins, 0]
xmaxs = vgtop[xindexmaxs, 0]
dx = np.abs(xmaxs - xmins)
dy = np.abs(ymaxs - ymins)
colors = ['white', 'white', 'white', 'tab:blue']

(vRtotal, vRtotal_sem, vRtotal_std) = find_means(vR, ([vR_ind[3]], [vR_ind[1]], [vR_ind[0]], [vR_ind[2]]))
vRtotal_tuple = (vRtotal[0, 0], vRtotal_sem[0, 0], vRtotal_std[0, 0])

for i in range(len(ymins)):
    ax.add_patch(Rectangle((xmins[i], ymaxs[i]), dx[i], dy[i], edgecolor=colors[i], fill=None, ls='--', lw=1))


#     print(vRtotal_tuple)
vR_tuple = find_means(vR, grid_bounds)
(meansR, semR, stdR) = vR_tuple
vT_tuple = find_means(vT, grid_bounds)
(meansT, semT, stdT) = vT_tuple
vs_tuple = find_means(vs, grid_bounds)
(meansS, semS, stdS) = vs_tuple



for i in range((len(ymins))):
    for j in range((len(ymins))):
        if i ==j:
            ax.text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % (meansR[i,j].item() / vRtotal.item()), 
                    horizontalalignment='center', verticalalignment='center', color=colors[i])
            # ax0.text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % (meansT[i,j] / meansS[i,j]), 
                #    horizontalalignment='center', verticalalignment='center', color='white')
        else:
            ax.text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % (meansR[i,j].item() / vRtotal.item()), 
                    horizontalalignment='center', verticalalignment='center', color='0.9')
            # ax0.text(np.mean((xmins[i], xmaxs[i])), np.mean((ymins[j], ymaxs[j])), "%.3f" % (meansT[i,j] / meansS[i,j]), 
                #    horizontalalignment='center', verticalalignment='center', color='white')


((t_vRvR, tsem_vRvR, tstd_vRvR), (t_vTvs, tsem_vTvs, tstd_vTvs), (t_vRvs, tsem_vRvs, tstd_vRvs), 
        (t_vTvRvT, tsem_vTvRvT, tstd_vTvRvT)) = calculate_transparencies(vR_tuple, vT_tuple, vs_tuple, vRtotal_tuple, nu, print_t=True)

if check_traces == 0:
    transparencies_dict['B='+str(field_val)+' T, nu<=2'] = (field_val, np.array([t_vRvR, t_vTvs, t_vRvs, t_vTvRvT]))
    sem_dict['B='+str(field_val)+' T, nu<=2'] = (field_val, np.array([tsem_vRvR, tsem_vTvs, tsem_vRvs, tsem_vTvRvT]))
    std_dict['B='+str(field_val)+' T, nu<=2'] = (field_val, np.array([tstd_vRvR, tstd_vTvs, tstd_vRvs, tstd_vTvRvT]))
    

f1_ax2.set_xlabel('top QPC (V)')
f1_ax2.set_ylabel('bottom QPC (V)')

skunk.connect(f1_ax1, 'full')
f1_ax1.set_axis_off()
skunk.connect(f1_ax3, 'zoom')
f1_ax3.set_axis_off()



svg = skunk.insert(
    {
        'full':'FIG2Apaper.svg',
        'zoom': 'FIG2C.svg'
    })


# fig1.canvas.draw()
# # we want the legend included in the bbox_inches='tight' calcs.
# # cbar1.set_in_layout(True)
# # cbar2.set_in_layout(True)
# # we don't want the layout to change at this point.
fig1.set_constrained_layout(True)
#fig1.tight_layout()

skunk.display(svg)
# cairosvg.svg2pdf(bytestring=svg, write_to='fig2.pdf')
# cairosvg.svg2eps(bytestring=svg, write_to='fig2.eps')
# fig1.savefig('Figure1.pdf', dpi=300)
# ax[0,2].axis('off')
# ax[1,2].axis('off')
# plt.tight_layout()

# plt.savefig("Figure4.eps")